# O7 — GSM gold-token degeneracy screen (Probe-3 mechanistic gate)

Probe-3 mechanistic was **excluded for Blocksworld** because the gold token was degenerate (only ~3 distinct gold tokens across the bank). **ALGO passed** (modal gold token 21.3%, under the 50% cutoff). **GSM was never screened.**

This notebook runs the **same degeneracy check** on GSM:

1. Extract the gold final-answer **content** for every GSM bank row (canonical + W1–W6) via the same `gsm_gold_content` helper as the mechanistic Colab (not `####` scaffolding).
2. Tokenize with each model tokenizer: `Qwen/Qwen2.5-1.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct`.
3. Resolve the **prompt-aware first BPE** of the gold content after the Appendix-N Probe-1 chat prompt (identical `resolve_target_token` logic).
4. Report: `# distinct` gold first tokens, modal token + share, Shannon entropy (bits).
5. **Cutoff (same as ALGO/BW):** FAIL if modal share **> 50%** on all items **or** on canonical-only.

**PASS** → GSM becomes a second mechanistic family; O8 runs on ALGO **and** GSM.  
**FAIL** → report in the measurement-failure table alongside BW.

**Note:** tokenizer-only — GPU not required (T4 ok; no model weights loaded).

**Outputs:** `O7_gsm_degeneracy_check.csv`, `O7_gsm_degeneracy_items.csv` (per-item audit), printed PASS/FAIL.


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## Load GSM bank + gold content (Appendix H: content, not format keywords)


In [ ]:
from __future__ import annotations

import json
import math
import re
from collections import Counter
from typing import Any

import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
]
DEGEN_FRAC = 0.5  # same cutoff as ALGO / BW mechanistic screen
VARIANTS = ("canonical", "W1", "W2", "W3", "W4", "W5", "W6")

PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)
GSM_FORMAT = (
    "Write the final numerical answer on its own line as #### <number>. "
    "No other text after that tag."
)
FORMAT_KEYWORDS = {
    "path", "count", "selected", "coins", "scoops", "total", "answer",
    "final", "####", "#", ":", "[", "]", "{", "}", ",",
    "path:", "count:", "selected:", "coins:", "scoops:",
}

GSM_BANK = REPO_ROOT / "data/problems/question_bank_gsm.csv"
O7_CSV = OUT_DIR / "O7_gsm_degeneracy_check.csv"
O7_ITEMS_CSV = OUT_DIR / "O7_gsm_degeneracy_items.csv"
O7_VERDICT_TXT = OUT_DIR / "O7_gsm_degeneracy_verdict.txt"

SUMMARY_COLUMNS = [
    "family",
    "model",
    "scope",  # all | canonical
    "n_items",
    "n_distinct_gold_first_tokens",
    "modal_gold_first_token",
    "modal_count",
    "modal_share",
    "entropy_bits",
    "degen_frac_cutoff",
    "degenerate",
    "verdict",  # PASS | FAIL
]


def _norm_vt(v: str) -> str:
    v = str(v).strip()
    return "canonical" if v.lower() == "canonical" else v.upper()


def _strip_csv_quotes(text: str) -> str:
    s = str(text)
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return s


def gsm_gold_content(correct_answer: str) -> str:
    """Numeric answer-content span. Never #### scaffolding (same as mech notebook)."""
    s = str(correct_answer).strip()
    s = re.sub(r"^####\s*", "", s)
    s = s.replace(",", "")
    try:
        f = float(s)
        if f == int(f):
            return str(int(f))
        return str(f)
    except ValueError:
        m = re.findall(r"-?\d+(?:\.\d+)?", s)
        if not m:
            raise ValueError(f"no numeric gold in {correct_answer!r}")
        return m[-1]


def build_user(problem_text: str) -> str:
    return PROBE1_TEMPLATE.format(
        problem=str(problem_text).strip(),
        family_specific_output_format=GSM_FORMAT,
    )


def load_gsm_items(limit: int | None) -> list[dict[str, Any]]:
    df = pd.read_csv(GSM_BANK, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    df["variant_type"] = df["variant_type"].map(_norm_vt)
    df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
    df["correct_answer"] = df["correct_answer"].map(_strip_csv_quotes)
    can_ids = set(df.loc[df.variant_type == "canonical", "problem_id"])
    items: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        pid = str(row["problem_id"])
        vt = str(row["variant_type"])
        if vt not in VARIANTS or pid not in can_ids:
            continue
        gold = gsm_gold_content(str(row["correct_answer"]))
        items.append(
            {
                "family": "GSM",
                "problem_id": pid,
                "variant": vt,
                "problem_text": str(row["problem_text"]),
                "correct_answer": str(row["correct_answer"]),
                "gold_content": gold,
            }
        )
    if limit is not None:
        keep_ids = sorted(can_ids)[:limit]
        keep = set(keep_ids)
        items = [x for x in items if x["problem_id"] in keep]
    return items


ITEMS = load_gsm_items(LIMIT)
print(f"[gsm] n_items={len(ITEMS)}  LIMIT={LIMIT}")
print(pd.DataFrame(ITEMS).groupby("variant").size().reindex(list(VARIANTS)).to_string())
_content_vc = Counter(x["gold_content"] for x in ITEMS)
_modal, _n = _content_vc.most_common(1)[0]
print(
    f"[gsm] gold_content (pre-tokenize) modal={_modal!r} "
    f"{_n}/{len(ITEMS)}={_n/len(ITEMS):.1%}  distinct={len(_content_vc)}"
)


## Prompt-aware first gold token + degeneracy / entropy

Same `resolve_target_token` as the mechanistic frequency-controlled notebook. Degenerate if modal decoded first-token share > 50% on **all items** or on **canonical-only**.


In [ ]:
def wrap_chat(tokenizer, user_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


def resolve_target_token(tokenizer, prompt: str, answer: str) -> tuple[int, str, list[int], str]:
    """Prompt-aware first token of `answer` after `prompt` (mechanistic scripts)."""

    def enc(text: str) -> list[int]:
        return tokenizer.encode(text, add_special_tokens=False)

    prompt_ids = enc(prompt)
    answer_ids_bare = enc(answer)
    candidates: list[tuple[str, int, int, list[int]]] = []
    for sep in ("", " "):
        joint = enc(prompt + sep + answer)
        if len(joint) <= len(prompt_ids):
            continue
        if joint[: len(prompt_ids)] != prompt_ids:
            continue
        rest = joint[len(prompt_ids) :]
        candidates.append((sep, int(rest[0]), len(joint), rest))
    if not candidates:
        if not answer_ids_bare:
            return -1, "", [], "EMPTY"
        tid = int(answer_ids_bare[0])
        return tid, tokenizer.decode([tid]), answer_ids_bare, "FALLBACK"
    candidates.sort(key=lambda c: c[2])
    sep, tid, _, rest = candidates[0]
    return tid, tokenizer.decode([tid]), rest, repr(sep)


def assert_content_gold(decoded: str) -> None:
    d = decoded.strip().lower()
    compact = d.replace(" ", "")
    if compact in FORMAT_KEYWORDS or d in FORMAT_KEYWORDS:
        raise AssertionError(f"gold token is a format keyword: {decoded!r}")
    if not re.search(r"\d", decoded):
        raise AssertionError(f"GSM content-gold token must contain a digit, got {decoded!r}")


def shannon_entropy_bits(tokens: list[str]) -> float:
    n = len(tokens)
    if n == 0:
        return float("nan")
    vc = Counter(tokens)
    ent = 0.0
    for c in vc.values():
        p = c / n
        ent -= p * math.log2(p)
    return float(ent)


def degeneracy_stats(tokens: list[str]) -> dict[str, Any]:
    n = len(tokens)
    if n == 0:
        return {
            "n_items": 0,
            "n_distinct_gold_first_tokens": 0,
            "modal_gold_first_token": "",
            "modal_count": 0,
            "modal_share": float("nan"),
            "entropy_bits": float("nan"),
            "degenerate": True,
        }
    vc = Counter(tokens)
    modal, n_m = vc.most_common(1)[0]
    share = n_m / n
    return {
        "n_items": n,
        "n_distinct_gold_first_tokens": len(vc),
        "modal_gold_first_token": modal,
        "modal_count": int(n_m),
        "modal_share": round(share, 6),
        "entropy_bits": round(shannon_entropy_bits(tokens), 6),
        "degenerate": bool(share > DEGEN_FRAC),
    }


def load_tokenizer(model_id: str):
    if DRY_RUN:
        # Offline pipeline check: whitespace / digit-aware fake BPE.
        class FakeTok:
            def apply_chat_template(self, messages, add_generation_prompt=True, tokenize=False):
                return "USER:" + messages[0]["content"] + "\nASSISTANT:"

            def encode(self, text, add_special_tokens=False):
                # Keep leading digits of numbers as separate "tokens" for a realistic screen.
                ids: list[int] = []
                i = 0
                s = str(text)
                while i < len(s):
                    if s[i].isdigit():
                        j = i
                        while j < len(s) and s[j].isdigit():
                            j += 1
                        # first-digit token id stable across numbers sharing a leading digit
                        ids.append(1000 + int(s[i]))
                        if j > i + 1:
                            ids.append(2000 + int(s[i + 1 : j] or "0") % 997)
                        i = j
                    else:
                        ids.append(10 + (ord(s[i]) % 200))
                        i += 1
                return ids

            def decode(self, ids):
                if not ids:
                    return ""
                tid = int(ids[0]) if isinstance(ids, list) else int(ids)
                if 1000 <= tid <= 1009:
                    return str(tid - 1000)
                return f"t{tid}"

        print(f"[tok] DRY_RUN fake tokenizer for {model_id}")
        return FakeTok()
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or True)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    print(f"[tok] loaded {model_id}")
    return tok


def score_items_for_model(model_id: str, items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    tok = load_tokenizer(model_id)
    rows: list[dict[str, Any]] = []
    for it in tqdm(items, desc=model_id.split("/")[-1]):
        user = build_user(it["problem_text"])
        prompt = wrap_chat(tok, user)
        tid, decoded, gold_ids, sep_note = resolve_target_token(tok, prompt, it["gold_content"])
        assert_content_gold(decoded)
        rows.append(
            {
                "family": "GSM",
                "model": model_id,
                "problem_id": it["problem_id"],
                "variant": it["variant"],
                "gold_content": it["gold_content"],
                "gold_first_token_id": int(tid),
                "gold_first_token_decoded": decoded,
                "n_gold_token_ids": len(gold_ids),
                "sep_note": sep_note,
            }
        )
    return rows


all_item_rows: list[dict[str, Any]] = []
summary_rows: list[dict[str, Any]] = []

for model_id in MODELS:
    rows = score_items_for_model(model_id, ITEMS)
    all_item_rows.extend(rows)
    dfm = pd.DataFrame(rows)
    scopes = {
        "all": dfm["gold_first_token_decoded"].astype(str).tolist(),
        "canonical": dfm.loc[
            dfm["variant"] == "canonical", "gold_first_token_decoded"
        ].astype(str).tolist(),
    }
    model_fail = False
    for scope, toks in scopes.items():
        st = degeneracy_stats(toks)
        verd = "FAIL" if st["degenerate"] else "PASS"
        if st["degenerate"]:
            model_fail = True
        summary_rows.append(
            {
                "family": "GSM",
                "model": model_id,
                "scope": scope,
                **{k: st[k] for k in (
                    "n_items",
                    "n_distinct_gold_first_tokens",
                    "modal_gold_first_token",
                    "modal_count",
                    "modal_share",
                    "entropy_bits",
                    "degenerate",
                )},
                "degen_frac_cutoff": DEGEN_FRAC,
                "verdict": verd,
            }
        )
    print(
        f"[{model_id}] model_verdict={'FAIL' if model_fail else 'PASS'} "
        f"(cutoff={DEGEN_FRAC})"
    )

items_df = pd.DataFrame(all_item_rows)
items_df.to_csv(O7_ITEMS_CSV, index=False)
summary_df = pd.DataFrame(summary_rows, columns=SUMMARY_COLUMNS)
summary_df.to_csv(O7_CSV, index=False)

overall_fail = bool(summary_df["degenerate"].astype(bool).any())
overall = "FAIL" if overall_fail else "PASS"

if overall == "PASS":
    paragraph = (
        f"VERDICT: PASS. GSM gold-first-token degeneracy is below the {DEGEN_FRAC:.0%} "
        f"modal-share cutoff used for ALGO/BW on both Qwen2.5-1.5B and Qwen2.5-3B "
        f"(see O7_gsm_degeneracy_check.csv). GSM becomes a second mechanistic family; "
        f"O8 should run on ALGO and GSM."
    )
else:
    fail_bits = summary_df[summary_df["degenerate"].astype(bool)][
        ["model", "scope", "modal_gold_first_token", "modal_share"]
    ]
    paragraph = (
        f"VERDICT: FAIL. GSM fails the same {DEGEN_FRAC:.0%} modal-share gold-token "
        f"degeneracy screen used for ALGO/BW on at least one (model, scope) cell "
        f"({fail_bits.to_dict(orient='records')}). Report GSM in the measurement-failure "
        f"table alongside BW; do not escalate GSM into O8 mechanistic readout."
    )

O7_VERDICT_TXT.write_text(paragraph + "\n")
print("\n=== O7_gsm_degeneracy_check.csv ===")
print(summary_df.to_string(index=False))
print("\n=== VERDICT ===")
print(paragraph)
print(f"\n[wrote] {O7_CSV}")
print(f"[wrote] {O7_ITEMS_CSV}")
print(f"[wrote] {O7_VERDICT_TXT}")


## Download / Drive backup

Copy after Colab:
- `O7_gsm_degeneracy_check.csv` → `results/derived/O7_gsm_degeneracy_check.csv`
- `O7_gsm_degeneracy_items.csv` → `results/raw/O7_gsm_degeneracy_items.csv`
- `O7_gsm_degeneracy_verdict.txt` → `results/derived/O7_gsm_degeneracy_verdict.txt`


In [ ]:
_out_files = [O7_CSV, O7_ITEMS_CSV, O7_VERDICT_TXT]
_drive_dir = Path("/content/drive/MyDrive/rvc_colab_out")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
